In [0]:

# Problem 1
#Symptom: Silver has exactly 4,074 rows — same as bronze. Finance reports only one product per order is showing in reports.
#Find the bug and fix it.

#Bug: dropDuplicates(["order_number"]) is called after explode — keeps only one product row per order, silently drops all others.


import pyspark.sql.functions as F

df_silver = spark.read.table("silver.sales_clean").filter(F.col("order_datetime").isNotNull())

df_daily = (
    df_silver
    .withColumn("order_date", F.col("order_datetime").cast("date"))
    .groupBy("order_date")
    .agg(
        F.sum(F.col("price") * F.col("qty")).alias("daily_revenue"),
        F.countDistinct("order_number").alias("order_count")
    )
)

(df_daily
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.daily_revenue"))

spark.sql("""
    SELECT order_date, SUM(daily_revenue) as revenue, SUM(order_count) as orders
    FROM gold.daily_revenue
    GROUP BY order_date
    ORDER BY order_date
""").show()







In [0]:
%sql

-- Problem 2 Performance Crisis in Gold
-- Scenario: gold.daily_revenue has 500 million rows across 3 years. This query takes 45 minutes:

SELECT 
    order_date,
    SUM(daily_revenue) as total_revenue,
    SUM(order_count)   as total_orders
FROM gold.daily_revenue
WHERE order_date BETWEEN '2022-01-01' AND '2022-03-31'
GROUP BY order_date
ORDER BY order_date

-- Three fixes proposed: sql-- Fix A  OPTIMIZE gold.daily_revenue ZORDER BY (daily_revenue)

-- Fix B OPTIMIZE gold.daily_revenue ZORDER BY (order_date)b

-- Fix C ALTER TABLE gold.daily_revenue ADD PARTITION (order_date)

-- OPTIMIZE gold.daily_revenue ZORDER BY (order_date)

In [0]:
# ============================================================
# Real World Problem 5 — Incorrect Join Producing Inflated Revenue
# Symptom: total_revenue is 3x higher than gold.product_revenue
# Root cause: joining on order_number only creates cartesian
# product within each order group, multiplying revenue by
# average number of clicked products per order
# ============================================================

import pyspark.sql.functions as F

df_clicks = spark.read.table("silver.clicks_clean")
df_sales  = spark.read.table("silver.sales_clean")

df_conversion = (
    df_clicks
    .join(
        df_sales,
        # FIX 1: join on both order_number AND product_id
        # order_number alone = cartesian product = inflated revenue
        (df_clicks["order_number"]       == df_sales["order_number"]) &
        (df_clicks["clicked_product_id"] == df_sales["product_id"]),
        "left"  # keep all clicks, NULL on sales side = not purchased
    )
    # FIX 2: group by clicked_product_id only
    # grouping by order_number too = one row per product per order (too granular)
    .groupBy(df_clicks["clicked_product_id"])
    .agg(
        # FIX 3: count not countDistinct
        # countDistinct(order_number) grouped by order_number always returns 1
        F.count(df_clicks["order_number"]).alias("total_clicks"),

        # revenue now correct — only counts sales rows that matched the join
        F.sum(F.col("price") * F.col("qty")).alias("total_revenue"),

        # isNotNull on sales side identifies converted rows after left join
        F.countDistinct(
            F.when(df_sales["product_id"].isNotNull(), df_clicks["order_number"])
        ).alias("converted_orders")
    )
    # conversion_rate = what % of clicks resulted in a purchase
    .withColumn(
        "conversion_rate",
        F.round(F.col("converted_orders") / F.col("total_clicks") * 100, 2)
    )
)

(df_conversion
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.click_conversion"))

spark.read.table("gold.click_conversion").show(10, truncate=False)